In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

# Load Environemnt Variable

In [3]:
HUGGINGFACEHUB_API_TOKEN=os.getenv("HUGGINGFACEHUB_API_TOKEN")
TAVILY_API_KEY=os.getenv("TAVILY_API_KEY")
LANGSMITH_TRACING=os.getenv("LANGSMITH_TRACING")
LANGSMITH_ENDPOINT=os.getenv("LANGSMITH_ENDPOINT")
LANGSMITH_API_KEY=os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT=os.getenv("LANGSMITH_PROJECT")
QDRANT_URL=os.getenv("QDRANT_URL")
QDRANT_API_KEY=os.getenv("QDRANT_API_KEY")

# Load HuggingFace Endpoints with API

In [6]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings
model = "sentence-transformers/all-mpnet-base-v2"
hf = HuggingFaceEndpointEmbeddings(
    model=model,
    task="feature-extraction",
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN,
)

result=hf.embed_query("hello world")

In [8]:
print(len(result))

768


In [10]:
print(result[:5])

[0.026249675080180168, 0.013395573012530804, -0.0045331381261348724, -0.021791430190205574, 0.054551851004362106]


# Load Groq LLM provider with API

In [11]:
from langchain_groq import ChatGroq

model=ChatGroq(
    model="openai/gpt-oss-20b"
)

In [12]:
model.invoke("how are you")

AIMessage(content='I’m doing great—thanks for asking! How can I help you today?', additional_kwargs={'reasoning_content': 'The user asks "how are you". We should respond politely. Probably a brief answer: "I\'m fine, thanks. How can I help you?" Let\'s comply.'}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 74, 'total_tokens': 133, 'completion_time': 0.05320699, 'prompt_time': 0.005489793, 'queue_time': 0.046984506, 'total_time': 0.058696783}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a5ac2a5d7b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--4fb003c2-944d-47c5-ae2d-f7143a9eacbc-0', usage_metadata={'input_tokens': 74, 'output_tokens': 59, 'total_tokens': 133})

# Setting up Qdrant online vectorstore with API

In [13]:
from qdrant_client import QdrantClient,models

client=QdrantClient(url=QDRANT_URL,api_key=QDRANT_API_KEY)

if client.collection_exists(collection_name="documents"):
    pass
else:
    client.create_collection(collection_name="documents",vectors_config=models.VectorParams(size=768,distance=models.Distance.COSINE))

In [14]:
from langchain_qdrant import Qdrant

qdrant_store=Qdrant.from_existing_collection(
    embedding=hf,
    collection_name="documents",
    url=QDRANT_URL
)

In [15]:
import uuid
from langchain.schema import Document

new_documents = [
    Document(id=str(uuid.uuid4()), page_content="LangChain is a framework that helps developers build applications powered by large language models using composable modules."),
    Document(id=str(uuid.uuid4()), page_content="Pinecone is a managed vector database that enables scalable similarity search and retrieval for machine learning applications."),
    Document(id=str(uuid.uuid4()), page_content="Embeddings are vector representations of text or data that capture semantic meaning and allow similarity comparisons."),
    Document(id=str(uuid.uuid4()), page_content="Docker is a platform that enables developers to package applications into containers for consistent deployment."),
    Document(id=str(uuid.uuid4()), page_content="Kubernetes is an open-source system for automating deployment, scaling, and management of containerized applications."),
    Document(id=str(uuid.uuid4()), page_content="Transformers are deep learning architectures that use self-attention mechanisms for processing sequential data like text."),
    Document(id=str(uuid.uuid4()), page_content="Natural Language Processing (NLP) is a field of AI focused on enabling computers to understand and generate human language."),
    Document(id=str(uuid.uuid4()), page_content="Elasticsearch is a distributed search and analytics engine commonly used for log analysis, full-text search, and data exploration."),
    Document(id=str(uuid.uuid4()), page_content="Hugging Face is a company and community providing tools, datasets, and models for natural language processing and machine learning."),
    Document(id=str(uuid.uuid4()), page_content="Neural networks are computational models inspired by the human brain that consist of layers of interconnected nodes (neurons).")
]


In [16]:
qdrant_store.add_documents(new_documents)

['4366650c-b2ee-43a6-abb2-db86b7441a8a',
 '26f269de-0094-4b48-b749-98f81713aa07',
 '5fa10c5d-38cb-440a-a82e-2b656b839908',
 '62927b52-1322-48cd-b54e-6fb396cd322a',
 '16eb1a98-f304-45f8-b612-07f7900f90a9',
 '83a06bf6-62b9-4657-a9b4-a3a8c5f4ad49',
 '78333f57-1802-4f19-8407-94c3c2d92de9',
 '53cf639e-7357-4429-a7b3-06598a068575',
 '5c5f6212-a87b-4344-8377-df6ecc4558ee',
 'ea1ef50e-7f74-42e9-b7bf-62faa51409a8']

In [17]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=20, chunk_overlap=10, separators=["\n\n", "\n", " ", ""]
)

text_splitter.split_text("Neural networks are computational models inspired by the human brain that consist of layers of interconnected nodes (neurons).")

['Neural networks are',
 'are computational',
 'models inspired by',
 'by the human brain',
 'brain that consist',
 'consist of layers',
 'of layers of',
 'of interconnected',
 'nodes (neurons).']

In [ ]:
from pydantic import BaseModel
import tempfile
from typing import List, Dict, Any

